# Notebook 08: Data Validation and Drift

## Why This Matters

Data quality is the silent killer of ML systems. Studies from industry consistently find that data issues—not model architecture—cause the majority of production ML failures. Google's foundational paper 'Hidden Technical Debt in Machine Learning Systems' specifically calls out data dependencies as the most dangerous form of technical debt. Understanding the taxonomy of drift (covariate shift, concept drift, label drift, dataset shift) and the statistical tools to detect them—PSI, KL divergence, the Kolmogorov-Smirnov test, Maximum Mean Discrepancy—is essential knowledge for any ML engineer who works on production systems. Tools like TensorFlow Data Validation, Great Expectations, and Evidently AI have made this a first-class engineering concern.

## 1. Taxonomy of Drift

Not all drift is the same. The type of drift determines the appropriate detection method and remediation strategy. Conflating them leads to wrong diagnoses.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Taxonomy of Distribution Drift in ML Systems', fontsize=14, fontweight='bold')

x = np.linspace(-4, 6, 500)

from scipy.stats import norm

# 1. Covariate Shift: P(X) changes, P(Y|X) stays the same
ax = axes[0][0]
ax.plot(x, norm.pdf(x, 0, 1), 'b-', linewidth=2.5, label='Training P(X)')
ax.plot(x, norm.pdf(x, 2, 1.2), 'r--', linewidth=2.5, label='Production P(X)')
ax.fill_between(x, norm.pdf(x, 0, 1), alpha=0.2, color='blue')
ax.fill_between(x, norm.pdf(x, 2, 1.2), alpha=0.2, color='red')
ax.set_title('1. Covariate Shift\nP(X) changes, P(Y|X) unchanged', fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlabel('Feature value X')
ax.set_ylabel('Density')
ax.grid(True, alpha=0.3)
ax.text(0, -0.06, 'Example: Training on US users, deployed to European users (different age distribution)',
        ha='center', fontsize=8, color='#555', style='italic')

# 2. Concept Drift: P(Y|X) changes, P(X) may or may not change  
ax2 = axes[0][1]
x_feat = np.linspace(0, 10, 200)

# Before: linear relationship
y_before = 0.5 * x_feat + 1 + np.random.normal(0, 0.3, len(x_feat))
y_after = -0.3 * x_feat + 8 + np.random.normal(0, 0.3, len(x_feat))

ax2.scatter(x_feat, y_before, alpha=0.4, color='blue', s=15, label='Training: P(Y|X) = positive slope')
ax2.scatter(x_feat, y_after, alpha=0.4, color='red', s=15, label='Production: P(Y|X) = negative slope')
ax2.set_title('2. Concept Drift\nP(Y|X) changes (relationship changes)', fontsize=10, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlabel('Feature value X')
ax2.set_ylabel('Label Y')
ax2.grid(True, alpha=0.3)
ax2.text(5, -0.3, 'Example: User behavior changes (pandemic → travel demand reverses)',
         ha='center', fontsize=8, color='#555', style='italic')

# 3. Label Drift: P(Y) changes
ax3 = axes[1][0]
categories = ['Label=0', 'Label=1']
train_dist = [0.90, 0.10]  # 10% positive in training
prod_dist = [0.70, 0.30]   # 30% positive in production (fraud spike)
x_pos = np.arange(len(categories))
ax3.bar(x_pos - 0.2, train_dist, width=0.35, color='blue', alpha=0.7, label='Training P(Y)')
ax3.bar(x_pos + 0.2, prod_dist, width=0.35, color='red', alpha=0.7, label='Production P(Y)')
ax3.set_title('3. Label Drift / Prior Shift\nP(Y) changes', fontsize=10, fontweight='bold')
ax3.legend(fontsize=9)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(categories)
ax3.set_ylabel('Proportion')
ax3.grid(True, alpha=0.3, axis='y')
ax3.text(0.5, -0.1, 'Example: Fraud rate spikes (10% → 30%), model calibration breaks',
         ha='center', fontsize=8, color='#555', style='italic')

# 4. Dataset Shift combined
ax4 = axes[1][1]
ax4.set_xlim(0, 10); ax4.set_ylim(0, 10); ax4.axis('off')
ax4.set_title('4. Drift Taxonomy Summary', fontsize=10, fontweight='bold')

drift_types = [
    ('Covariate Shift', '#2980B9', 'P(X) changes\nP(Y|X) same', 'Feature distribution changes\n(detectable via feature monitoring)'),
    ('Concept Drift', '#E74C3C', 'P(Y|X) changes\nP(X) may be same', 'World changes so features\nmean something different'),
    ('Label Shift', '#8E44AD', 'P(Y) changes\nP(X|Y) may be same', 'Class proportions change\n(prior probability shift)'),
    ('Dataset Shift', '#E67E22', 'P(X,Y) changes', 'Combination of above\n(most general case)'),
]

for i, (name, color, formal, informal) in enumerate(drift_types):
    y = 8 - i * 2.2
    r = FancyBboxPatch((0.2, y - 0.5), 9.6, 1.8, boxstyle="round,pad=0.1",
                       facecolor=color, alpha=0.15, edgecolor=color, linewidth=1.5)
    ax4.add_patch(r)
    ax4.text(0.5, y + 0.55, name, fontsize=9, fontweight='bold', color=color)
    ax4.text(0.5, y + 0.1, formal, fontsize=8.5, color='#333')
    ax4.text(5.5, y + 0.3, informal, fontsize=8, color='#555', multialignment='left')

plt.tight_layout()
plt.savefig('drift_taxonomy.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Statistical Tests for Drift Detection

Different statistical tests are suited for different feature types and use cases. Understanding when to use each test—and what their limitations are—is important for building robust monitoring systems.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

def kolmogorov_smirnov_test(ref, curr):
    """KS test: non-parametric, good for continuous features."""
    stat, p_val = stats.ks_2samp(ref, curr)
    return {'statistic': stat, 'p_value': p_val, 'drift': p_val < 0.05}

def population_stability_index(expected, actual, n_bins=10):
    """PSI: industry standard, especially in financial ML."""
    bins = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf; bins[-1] = np.inf
    exp = np.histogram(expected, bins=bins)[0]
    act = np.histogram(actual, bins=bins)[0]
    exp = exp / (exp.sum() + 1e-10)
    act = act / (act.sum() + 1e-10)
    # Avoid log(0)
    exp = np.where(exp == 0, 1e-6, exp)
    act = np.where(act == 0, 1e-6, act)
    psi = np.sum((act - exp) * np.log(act / exp))
    severity = 'No drift' if psi < 0.1 else 'Monitor closely' if psi < 0.2 else 'Significant drift'
    return {'psi': psi, 'severity': severity, 'drift': psi > 0.1}

def chi_squared_test(ref_counts, curr_counts):
    """Chi-squared: for categorical features."""
    ref_pct = ref_counts / ref_counts.sum()
    expected = ref_pct * curr_counts.sum()
    stat, p_val = stats.chisquare(curr_counts, f_exp=expected)
    return {'statistic': stat, 'p_value': p_val, 'drift': p_val < 0.05}

def jensen_shannon_divergence(p, q):
    """JS divergence: symmetric version of KL, bounded [0,1]."""
    p = np.array(p) + 1e-10
    q = np.array(q) + 1e-10
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    jsd = 0.5 * (kl_pm + kl_qm)
    return {'jsd': jsd, 'severity': 'No drift' if jsd < 0.05 else 'Monitor' if jsd < 0.1 else 'Drift', 'drift': jsd > 0.05}


# Simulate reference and drifted production data
ref = np.random.normal(0, 1, 10000)

drift_scenarios = [
    ('No drift', np.random.normal(0, 1, 5000)),
    ('Small mean shift (+0.5σ)', np.random.normal(0.5, 1, 5000)),
    ('Moderate mean shift (+1σ)', np.random.normal(1.0, 1, 5000)),
    ('Large mean shift (+2σ)', np.random.normal(2.0, 1, 5000)),
    ('Variance change (2×)', np.random.normal(0, 2, 5000)),
    ('Bimodal production', np.concatenate([np.random.normal(-1, 0.5, 2500), np.random.normal(2, 0.5, 2500)])),
]

print(f"{'Scenario':35s} {'KS p-val':>10} {'KS drift':>9} {'PSI':>8} {'PSI sev':>18}")
print('-' * 85)

for name, curr in drift_scenarios:
    ks = kolmogorov_smirnov_test(ref, curr)
    psi_res = population_stability_index(ref, curr)
    print(f"{name:35s} {ks['p_value']:>10.4f} {'✓ YES' if ks['drift'] else '✗ NO':>9} "
          f"{psi_res['psi']:>8.4f} {psi_res['severity']:>18}")

print("\nPSI Interpretation Guide:")
print("  PSI < 0.1:  No significant drift, model stable")
print("  PSI 0.1-0.2: Some shift, monitor closely, consider retraining")
print("  PSI > 0.2:  Significant shift, retrain required")
print("\nNote: KS test detects any distributional difference.")
print("      PSI quantifies the magnitude — more actionable for alerts.")

## 3. Data Validation Schemas

Proactive schema validation catches data quality issues before they reach the model. TensorFlow Data Validation (TFDV) introduced the concept of a feature schema that specifies expected types, ranges, value sets, and missingness constraints.

In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional, Any

@dataclass
class FeatureSchema:
    name: str
    dtype: str           # 'float', 'int', 'str', 'bool'
    nullable: bool = False
    min_value: Optional[float] = None
    max_value: Optional[float] = None
    allowed_values: Optional[List[Any]] = None
    max_fraction_missing: float = 0.05
    description: str = ''

class DataValidator:
    """Schema-based data validator for ML pipelines."""
    
    def __init__(self, schema: List[FeatureSchema]):
        self.schema = {s.name: s for s in schema}
        self.violations = []
    
    def validate(self, df: pd.DataFrame) -> bool:
        self.violations = []
        self._check_required_columns(df)
        for col, spec in self.schema.items():
            if col not in df.columns:
                continue
            self._check_dtype(df, col, spec)
            self._check_nulls(df, col, spec)
            self._check_range(df, col, spec)
            self._check_allowed_values(df, col, spec)
        return len([v for v in self.violations if v['severity'] == 'ERROR']) == 0
    
    def _check_required_columns(self, df):
        for col in self.schema:
            if col not in df.columns:
                self.violations.append({'col': col, 'severity': 'ERROR',
                                        'issue': 'Required column missing'})
    
    def _check_dtype(self, df, col, spec):
        type_map = {'float': (float, np.float32, np.float64),
                    'int': (int, np.int32, np.int64),
                    'str': (str, object),
                    'bool': (bool, np.bool_)}
        expected = type_map.get(spec.dtype, ())
        if expected and not any(np.issubdtype(df[col].dtype, t) for t in [np.floating, np.integer, np.str_, np.bool_]
                                if spec.dtype in str(t)):
            pass  # simplified check
    
    def _check_nulls(self, df, col, spec):
        null_frac = df[col].isna().mean()
        if not spec.nullable and null_frac > 0:
            sev = 'ERROR' if null_frac > 0.1 else 'WARNING'
            self.violations.append({'col': col, 'severity': sev,
                                    'issue': f'{null_frac:.1%} nulls (not allowed)'})
        elif spec.nullable and null_frac > spec.max_fraction_missing:
            self.violations.append({'col': col, 'severity': 'WARNING',
                                    'issue': f'{null_frac:.1%} nulls > threshold {spec.max_fraction_missing:.1%}'})
    
    def _check_range(self, df, col, spec):
        if spec.min_value is not None:
            violations = (df[col] < spec.min_value).sum()
            if violations > 0:
                self.violations.append({'col': col, 'severity': 'ERROR',
                                        'issue': f'{violations} values below min {spec.min_value}'})
        if spec.max_value is not None:
            violations = (df[col] > spec.max_value).sum()
            if violations > 0:
                self.violations.append({'col': col, 'severity': 'ERROR',
                                        'issue': f'{violations} values above max {spec.max_value}'})
    
    def _check_allowed_values(self, df, col, spec):
        if spec.allowed_values:
            unknown = ~df[col].isin(spec.allowed_values + [None, np.nan])
            n_unknown = unknown.sum()
            if n_unknown > 0:
                new_vals = df[col][unknown].unique()[:5]
                self.violations.append({'col': col, 'severity': 'WARNING',
                                        'issue': f'{n_unknown} unknown values: {new_vals}'})
    
    def report(self):
        if not self.violations:
            print("  ✓ All schema checks passed")
            return
        errors = [v for v in self.violations if v['severity'] == 'ERROR']
        warnings = [v for v in self.violations if v['severity'] == 'WARNING']
        for v in self.violations:
            icon = '✗' if v['severity'] == 'ERROR' else '⚠'
            print(f"  {icon} [{v['severity']:7s}] {v['col']}: {v['issue']}")
        print(f"  Summary: {len(errors)} errors, {len(warnings)} warnings")


# Define a schema for a recommendation system's features
schema = [
    FeatureSchema('user_id', 'int', nullable=False, description='User identifier'),
    FeatureSchema('user_age', 'float', nullable=True, min_value=13, max_value=120,
                  max_fraction_missing=0.1, description='User age in years'),
    FeatureSchema('user_7d_ctr', 'float', nullable=True, min_value=0.0, max_value=1.0,
                  description='7-day click-through rate'),
    FeatureSchema('device_type', 'str', nullable=False,
                  allowed_values=['mobile', 'desktop', 'tablet', 'tv'],
                  description='Device type'),
    FeatureSchema('clicked', 'int', nullable=False, min_value=0, max_value=1,
                  description='Binary label'),
]

validator = DataValidator(schema)

# Clean data
np.random.seed(42)
n = 1000
clean_df = pd.DataFrame({
    'user_id': range(n),
    'user_age': np.random.normal(35, 10, n).clip(13, 90),
    'user_7d_ctr': np.random.beta(1, 10, n),
    'device_type': np.random.choice(['mobile', 'desktop', 'tablet'], n),
    'clicked': np.random.binomial(1, 0.05, n),
})

# Corrupt data
corrupt_df = clean_df.copy()
corrupt_df.loc[:50, 'user_age'] = np.nan  # 5% missing
corrupt_df.loc[100:110, 'user_age'] = 200  # out of range
corrupt_df.loc[200:220, 'device_type'] = 'smartwatch'  # new category
corrupt_df.loc[300:310, 'user_7d_ctr'] = -0.1  # below min

print("=" * 60)
print("Validating CLEAN data:")
print("=" * 60)
clean_ok = validator.validate(clean_df)
validator.report()
print(f"  Pipeline OK: {clean_ok}")

print("\n" + "=" * 60)
print("Validating CORRUPT data:")
print("=" * 60)
corrupt_ok = validator.validate(corrupt_df)
validator.report()
print(f"  Pipeline OK: {corrupt_ok}")

## 4. Handling Drift: Remediation Strategies

Detecting drift is only half the problem. The response strategy depends on the type and severity of drift, available labels, and the cost of retraining.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 10))
ax.set_xlim(0, 15); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Drift Remediation Decision Tree', fontsize=14, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=9):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        ax.text((x1+x2)/2+0.1, (y1+y2)/2, label, fontsize=8, color=color)

# Root
box(ax, 5.5, 8.5, 4, 1.2, 'Drift Detected', '#E74C3C')

# Branch: data quality vs distribution
box(ax, 0.5, 6.5, 4, 1.2, 'Data Quality Issue?\n(pipeline, schema)', '#E67E22')
box(ax, 8, 6.5, 4, 1.2, 'Statistical\nDistribution Drift?', '#8E44AD')

arr(ax, 7.5, 8.5, 2.5, 7.7, 'Yes')
arr(ax, 7.5, 8.5, 10, 7.7, 'No')

# Data quality responses
box(ax, 0.2, 4.5, 2.2, 1.5, 'Fix Upstream\nPipeline', '#2980B9')
box(ax, 2.8, 4.5, 2.2, 1.5, 'Update Feature\nImputation', '#3498DB')

arr(ax, 2.5, 6.5, 1.3, 6.0)
arr(ax, 2.5, 6.5, 3.9, 6.0)

# Drift type assessment
box(ax, 6, 4.5, 2.5, 1.5, 'Covariate\nShift?\nP(X) drifted', '#8E44AD')
box(ax, 9.5, 4.5, 2.5, 1.5, 'Concept\nDrift?\nP(Y|X) changed', '#E74C3C')

arr(ax, 10, 6.5, 7.25, 6.0)
arr(ax, 10, 6.5, 10.75, 6.0)

# Covariate shift responses
box(ax, 4.5, 2.5, 3, 1.5, 'Importance\nWeighted\nRetraining', '#16A085')
box(ax, 8, 2.5, 3, 1.5, 'Trigger Full\nRetraining on\nNew Data', '#27AE60')
box(ax, 11.5, 2.5, 3, 1.5, 'Adjust Prior /\nRecalibrate\nOutput Layer', '#2980B9')

arr(ax, 7.25, 4.5, 6.0, 4.0)
arr(ax, 10.75, 4.5, 9.5, 4.0)
arr(ax, 10.75, 4.5, 13, 4.0)

ax.text(7.5, 1.5, 'General principle: Covariate shift → weight correction; Concept drift → new labels → retrain',
        ha='center', fontsize=9, color='#333', style='italic')

plt.tight_layout()
plt.savefig('drift_remediation.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nRemediation strategy summary:")
strategies = [
    ("Data pipeline fix", "Schema changed, feature unavailable", "Fix upstream source"),
    ("Imputation update", "Feature missing rate changed", "Update imputation strategy"),
    ("Importance weighting", "Covariate shift, P(X) changed", "Reweight training samples to match production"),
    ("Full retrain", "Concept drift, P(Y|X) changed", "Collect new labeled data, retrain from recent window"),
    ("Recalibration only", "Label prior shifted, P(Y) changed", "Adjust model output thresholds or Platt scaling"),
    ("Warm-start retrain", "Gradual drift, cost-sensitive", "Fine-tune existing model on recent data window"),
]
print(f"{'Strategy':25s} {'When':35s} {'How':40s}")
print('-' * 100)
for s, when, how in strategies:
    print(f"  {s:23s} {when:33s} {how}")

## Summary

| Drift Type | Formal Definition | Detection Method | Remediation |
|------------|-------------------|-----------------|-------------|
| Covariate Shift | P(X) changes, P(Y\|X) same | PSI, KS test on features | Importance weighting |
| Concept Drift | P(Y\|X) changes | Rolling accuracy on labeled data | Full retrain on new data |
| Label Shift | P(Y) changes | Monitor prediction rate, label distribution | Recalibrate, adjust priors |
| Dataset Shift | P(X,Y) changes | Combine above methods | Varies by cause |

**Statistical Tests**
- KS test: Non-parametric, detects any distributional difference in continuous features
- PSI: Quantifies magnitude of shift; industry standard (finance/ML); < 0.1 = stable, > 0.2 = retrain
- Chi-squared: Categorical feature drift detection
- Jensen-Shannon Divergence: Symmetric, bounded, useful for probability distributions

**Key Principles**
- Validate data schema BEFORE training (not just at serving time)
- Monitor feature distributions separately from model performance
- Label delay creates a monitoring gap — use proxy metrics and statistical monitors to bridge it
- Different features drift at different rates — prioritize high-cardinality and high-importance features